# Amparex (SDAX) — wie viel Customer Support kann KI übernehmen?

Diese Analyse klassifiziert jedes SDAX-Ticket danach, **wie** es gelöst wurde, und leitet
daraus ab, welcher Anteil des Support-Aufkommens von KI unterstützt oder ganz
übernommen werden könnte.

**Drei Auflösungs-Buckets** (plus `unclear`, wenn der Text nicht ausreicht):

| Bucket | Bedeutung |
|---|---|
| `question_answer` | Frage, die aus Doku/Produktwissen beantwortbar ist — nichts muss geändert werden |
| `support_todo` | Konkrete Routine-Tätigkeit eines Support-Mitarbeiters im System |
| `team_handoff` | Braucht ein anderes Team: Bug, Feature, DB-Fix, Hardware, externer Dienstleister |

Dazu kommen pro Ticket:

- **`sub_bucket`** — feinere Aufteilung innerhalb des Buckets
- **`intent_label`** — freier kanonischer Intent in 2–5 deutschen Wörtern. Die *kleineren*
  Buckets werden nicht vorab geraten, sondern durch Aggregation dieses Labels entdeckt.
- **`ai_mode`** — `fully_automatable` / `ai_drafts_human_sends` / `ai_assisted_triage` / `human_only`.
  Bewusst **getrennt** vom Bucket: ein `support_todo` kann voll automatisierbar sein,
  während eine `question_answer` zur Kassenabrechnung trotzdem einen Menschen braucht.
- **`confidence`** und **`reasoning`** zum Stichproben-Check.

**Ablauf:** Refresh (1,5 Jahre SDAX) → Aufbereitung → Pilot (60 Tickets) → Vollklassifikation
→ Validierung gegen harte Eskalations-Signale → Auswertung → Management-Summary.

**Voraussetzungen:** `.env` im Projektverzeichnis mit

```
JIRA_URL= / JIRA_USERNAME= / JIRA_PASSWORD=
AZURE_OPENAI_API_KEY=
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT_NAME=<Name des Deployments, nicht des Modells>
AZURE_OPENAI_API_VERSION=2024-10-21   # optional
```

Das Deployment muss auf einem Modell mit Structured Outputs laufen
(`gpt-4o-2024-08-06` oder neuer, `gpt-4o-mini` eingeschlossen). Ältere Deployments
lehnen das JSON-Schema ab — dann kommt jedes Ticket als `bad_request` zurück.

In [ ]:
import hashlib
import json
import subprocess
import sys
from datetime import UTC, date, datetime
from pathlib import Path

# A foreign working directory supplies REPO_ROOT explicitly (see docs/analysis.md).
REPO_ROOT = Path(globals().get("REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "settings.py").is_file():
    raise RuntimeError("Set REPO_ROOT to the repository directory before running.")
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import Markdown, display

from analysis_workflow import AnalysisSpec, analysis_manifest, select_cohort
from classification_cache import configuration_fingerprint
from reporting_policy import BERLIN
from settings import load_settings
from ticket_classifier import classifier_configuration, classify_tickets

pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 140)
settings = load_settings()
PROJECT = globals().get("PROJECT", "SDAX")
START_DATE = globals().get("START_DATE", date(2025, 1, 1))
END_DATE = globals().get("END_DATE", datetime.now(BERLIN).date())
REFRESH = globals().get("REFRESH", False)
RESUME = globals().get("RESUME", True)
RUN_CLASSIFICATION = globals().get("RUN_CLASSIFICATION", False)
OUTPUT_DIR = globals().get("OUTPUT_DIR", None)
classification_runner = globals().get("classification_runner", classify_tickets)
configuration = globals().get("configuration", classifier_configuration(settings))
spec = AnalysisSpec(PROJECT, START_DATE, END_DATE, configuration.policy_version)
classification_path = settings.paths.classifications
print(
    f"{PROJECT}: {START_DATE} bis {END_DATE}, REFRESH={REFRESH}, "
    f"RUN_CLASSIFICATION={RUN_CLASSIFICATION}"
)

## 1 · Refresh — 1,5 Jahre SDAX aus Jira ziehen

Der Pull läuft über `backfill_jira.py` statt über eigenen Fetch-Code im Notebook. Das Skript
existiert genau für diesen Fall: es checkpointet die Rohdaten nach jedem Projekt, sichert die
bestehende `data/jira_data.pkl` vor dem Schreiben, und merged per Upsert — bereits
gesammelte Tickets gehen also nicht verloren.

Die Ausgabe wird live durchgereicht. Bei einem Abbruch einfach die Zelle erneut ausführen
(`RESUME = True` nutzt den Checkpoint und holt nur das fehlende Intervall nach).

In [ ]:
if REFRESH:
    command = [
        sys.executable,
        str(REPO_ROOT / "backfill_jira.py"),
        "--project",
        PROJECT,
        "--start",
        START_DATE.isoformat(),
        "--end",
        END_DATE.isoformat(),
    ]
    if RESUME:
        command.append("--resume")
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("REFRESH=False: vorhandenen Cache offline lesen.")

## 2 · Laden und aufbereiten

Aus dem gemeinsamen Cache (SDIPR + SDAX + SDEU) nur die Amparex-Tickets herausziehen und
prüfen, wie viel Text überhaupt zur Verfügung steht. Das ist keine Formalie: Kommentare sind
das aussagekräftigste Feld, fehlen aber auf einem großen Teil der Tickets.

In [ ]:
from data_loading import load_data_snapshot
from file_io import read_file_snapshot

df_all, revision = load_data_snapshot(path=settings.paths.tickets, settings=settings)
if df_all is None or df_all.empty:
    raise RuntimeError(
        "Ticket cache is empty or missing; provide a cache before analysis."
    )
source_bytes, source_revision = read_file_snapshot(settings.paths.tickets)
if source_revision != revision:
    raise RuntimeError("cache_changed_during_analysis_load")
extraction_digest = hashlib.sha256(source_bytes).hexdigest()
del source_bytes
# Timestamp of this coherent local extraction, not a claim about Jira freshness.
extracted_at = datetime.now(UTC)
df = select_cohort(df_all, spec)
manifest = analysis_manifest(
    df,
    spec,
    extracted_at=extracted_at,
    extraction_digest=extraction_digest,
    classifier_fingerprint=configuration_fingerprint(configuration),
)
print(json.dumps(manifest, indent=2))

Kategorien kommen ausschliesslich aus der gemeinsamen Cache-Normalisierung. Moderne unbekannte Assets bleiben unbekannt.


In [ ]:
# Shared cache normalization owns category policy, including legacy-only mapping.
# Modern unresolved or foreign-workspace objects stay unknown. Never backfill here.
for label_column in ("Hauptkategorie", "Unterkategorie"):
    unknown = df[label_column].isin(["", "Unbekannt", "NA"]).sum()
    print(
        f"{label_column}: {unknown} unbekannt, {df[label_column].nunique()} verschiedene"
    )

In [ ]:
def fill_rate(series):
    """Report the populated share of a normalized text column."""
    text = series.astype(str).str.strip()
    return (~text.isin(["", "nan", "None", "[]", "NA"])).mean()


quality = pd.DataFrame(
    {
        "Feld": [
            "summary",
            "description",
            "comments",
            "Hauptkategorie",
            "Unterkategorie",
        ],
        "befuellt": [
            fill_rate(df[column])
            for column in [
                "summary",
                "description",
                "comments",
                "Hauptkategorie",
                "Unterkategorie",
            ]
        ],
        "Median Zeichen": [
            df[column].astype(str).str.len().median()
            for column in [
                "summary",
                "description",
                "comments",
                "Hauptkategorie",
                "Unterkategorie",
            ]
        ],
    }
)
quality["befuellt"] = (quality["befuellt"] * 100).round(1).astype(str) + " %"
display(quality)

print(f"\nHauptkategorien: {df['Hauptkategorie'].nunique()}")
print(f"Unterkategorien: {df['Unterkategorie'].nunique()}")

In [ ]:
from ticket_classifier import build_ticket_text

# Aggregate diagnostics only; do not save example ticket text into notebook outputs.
lengths = pd.Series(
    [len(build_ticket_text(row)) for _, row in df.iterrows()], dtype="int64"
)
print("Laenge des Klassifikations-Texts (Zeichen):")
print(lengths.describe().round(0).to_string())

## 3 · Pilot — 60 Tickets, quer über alle Hauptkategorien

Bevor die ~10.000 Tickets durchlaufen, ein stratifizierter Pilot: zwei Tickets je
Hauptkategorie. Damit lässt sich Taxonomie und Prompt prüfen, ohne den vollen Lauf zu
bezahlen. Die `reasoning`-Spalte ist genau dafür da — kurz durchlesen, bevor es weitergeht.

In [ ]:
# Preview matching results only. The single explicit paid-work step is below;
# it shares the same identity-aware cache, so a preview never duplicates model work.
PER_CATEGORY = 2
pilot_keys = (
    df.groupby("Hauptkategorie", dropna=False)["key"]
    .apply(lambda keys: keys.sample(min(len(keys), PER_CATEGORY), random_state=0))
    .tolist()
)
pilot = df[df["key"].isin(pilot_keys)].reset_index(drop=True)
pilot_results = classify_tickets(
    pilot, cache_path=classification_path, configuration=configuration, limit=0
)
print(f"Pilot: {len(pilot)} ausgewaehlt, {len(pilot_results)} passende Ergebnisse")

In [ ]:
pilot_view = pilot.merge(pilot_results, on="key", how="left", validate="one_to_one")
print(pilot_view["resolution_bucket"].value_counts(dropna=False).to_string())
print(pilot_view["ai_mode"].value_counts(dropna=False).to_string())

## 4 · Vollklassifikation

Läuft über alle SDAX-Tickets. Ergebnisse landen in `data/sdax_classifications.parquet` und
werden alle 100 Tickets weggeschrieben — ein Abbruch kostet also höchstens die letzten 100.
Ein erneuter Lauf klassifiziert nur, was neu ist oder dessen Text sich geändert hat.

**Größenordnung — gemessen, nicht geschätzt.** Ein Probelauf über 12 typische Tickets auf
dem Deployment `gpt-5.6-luna` ergab im Mittel **1.023 Input-** und **182 Output-Token** pro
Ticket (davon 103 Reasoning-Token). Hochgerechnet auf ~35.000 Tickets:

| | |
|---|---|
| Input | ~35,8 Mio. Token |
| Output | ~6,4 Mio. Token |
| Laufzeit bei 16 Workern | ~2,5 Stunden |

Die Kosten hängen am Azure-Tarif für dieses Deployment — die Token-Mengen oben mit der
eigenen Preisliste multiplizieren. Der grösste Hebel wäre der System-Prompt: er macht rund
800 der 1.023 Input-Token aus. Azure cached Prefixes erst ab 1.024 Token automatisch, der
Prompt liegt also knapp darunter.

Die Zelle unten gibt vor dem Start die geschätzte Token-Zahl aus, und `classify_tickets`
meldet nach jedem Lauf den tatsächlichen Verbrauch — nach dem Pilot also gegenrechnen,
bevor der Volllauf startet.

In [ ]:
if RUN_CLASSIFICATION:
    results = classification_runner(
        df, cache_path=classification_path, configuration=configuration, max_workers=16
    )
else:
    results = classify_tickets(
        df, cache_path=classification_path, configuration=configuration, limit=0
    )
print(f"{len(results):,} passende Ergebnisse fuer die ausgewaehlten Eingaben")

In [ ]:
from ticket_classifier import AI_MODES, RESOLUTION_BUCKETS

labelled = df.merge(
    results.drop(columns=["model", "text_hash"], errors="ignore"),
    on="key",
    how="left",
    validate="one_to_one",
)
BUCKET_ORDER = list(RESOLUTION_BUCKETS)
AI_MODE_ORDER = list(AI_MODES)
labelled["resolution_bucket"] = pd.Categorical(
    labelled["resolution_bucket"], categories=BUCKET_ORDER, ordered=True
)
labelled["ai_mode"] = pd.Categorical(
    labelled["ai_mode"], categories=AI_MODE_ORDER, ordered=True
)
success = labelled["error"].isna() & labelled["resolution_bucket"].notna()
print(
    f"{int(success.sum()):,} erfolgreich klassifiziert, "
    f"{int((~success).sum()):,} ohne aktuelles Ergebnis"
)

## 5 · Validierung gegen harte Signale

Die Klassifikation ist ein Modell-Urteil und muss gegengeprüft werden. Im Datensatz stecken
drei unabhängige, *strukturelle* Eskalations-Signale, die niemand für diese Analyse erzeugt
hat:

- **`has_exax_clone`** — Ticket wurde ins EXAX-Projekt geklont, also an die Entwicklung übergeben
- **`status`** `Frage an Fachabteilung` / `Antwort von Fachabteilung` — an eine Fachabteilung eskaliert
- **`resolution_band`** — wie lange es tatsächlich gedauert hat

Wenn `team_handoff` etwas taugt, muss die Klon-Rate in diesem Bucket deutlich höher liegen
als in den anderen. Tut sie das nicht, stimmt die Klassifikation nicht.

In [ ]:
# Richtungspruefung anhand normalisierter Klonprojekte.
ESCALATION_PROJECTS = ("AXT", "EXAX")
clone_targets = labelled["clone_in_project"].fillna("").astype(str)
labelled["cloned_to_dev"] = clone_targets.str.contains(
    "|".join(ESCALATION_PROJECTS), regex=True
)
labelled["escalated_structurally"] = labelled["cloned_to_dev"] | labelled[
    "status"
].isin(["Frage an Fachabteilung", "Antwort von Fachabteilung"])

print("Struktuelle Eskalations-Signale im Datensatz:")
print(
    f"  Klon nach {'/'.join(ESCALATION_PROJECTS)}: {labelled['cloned_to_dev'].sum():,}"
)
department_count = (
    labelled["status"]
    .isin(["Frage an Fachabteilung", "Antwort von Fachabteilung"])
    .sum()
)
print(f"  Status Fachabteilung: {department_count:,}")
print(
    f"  gesamt:                    {labelled['escalated_structurally'].sum():,} "
    f"({labelled['escalated_structurally'].mean():.1%} der Tickets)"
)
print(
    "\nDas Signal ist duenn (~1-2 %), taugt also als Richtungspruefung, nicht als\n"
    "Ground Truth: team_handoff muss klar oben liegen, exakte Werte sind Rauschen."
)

validation = (
    labelled.groupby("resolution_bucket", observed=False)
    .agg(
        Tickets=("key", "size"),
        Dev_Klon=("cloned_to_dev", "mean"),
        Fachabteilung=(
            "status",
            lambda s: s.isin(
                ["Frage an Fachabteilung", "Antwort von Fachabteilung"]
            ).mean(),
        ),
        strukturell_eskaliert=("escalated_structurally", "mean"),
        Median_Konfidenz=("confidence", "median"),
    )
    .reset_index()
)
for column in ["Dev_Klon", "Fachabteilung", "strukturell_eskaliert"]:
    validation[column] = (validation[column] * 100).round(1)

display(validation)

print(
    "\nLesart: die Zeile team_handoff muss bei 'strukturell_eskaliert' deutlich ueber den\n"
    "anderen Zeilen liegen. Ist der Unterschied klein, traegt die Klassifikation nicht."
)

In [ ]:
band_column = (
    "resolution_band" if "resolution_band" in labelled.columns else "resolution"
)
duration = (
    pd.crosstab(labelled["resolution_bucket"], labelled[band_column], normalize="index")
    * 100
).round(1)
display(duration)

if not duration.empty:
    figure = px.imshow(
        duration,
        text_auto=True,
        aspect="auto",
        color_continuous_scale="Blues",
        labels={"x": "Bearbeitungsdauer", "y": "Bucket", "color": "% der Zeile"},
        title="Bearbeitungsdauer je Auflösungs-Bucket (%, zeilenweise)",
    )
    figure.show()

In [ ]:
# No ticket text/reasoning samples in saved outputs; aggregate confidence only.
labelled_ok = labelled.dropna(subset=["resolution_bucket"])
display(
    labelled_ok.groupby("resolution_bucket", observed=True)["confidence"].describe()
)

## 6 · Auswertung

In [ ]:
bucket_counts = (
    labelled["resolution_bucket"]
    .value_counts(dropna=False)
    .rename_axis("Bucket")
    .reset_index(name="Tickets")
)
bucket_counts["Anteil %"] = (
    bucket_counts["Tickets"] / max(len(labelled), 1) * 100
).round(1)
display(bucket_counts)

figure = px.bar(
    bucket_counts,
    x="Bucket",
    y="Tickets",
    text="Anteil %",
    title=f"Auflösungs-Buckets über {len(labelled):,} SDAX-Tickets",
)
figure.update_traces(texttemplate="%{text} %", textposition="outside")
figure.show()

In [ ]:
ai_counts = (
    labelled["ai_mode"]
    .value_counts(dropna=False)
    .rename_axis("ai_mode")
    .reset_index(name="Tickets")
)
ai_counts["Anteil %"] = (ai_counts["Tickets"] / max(len(labelled), 1) * 100).round(1)
display(ai_counts)

figure = px.bar(
    ai_counts,
    x="ai_mode",
    y="Tickets",
    text="Anteil %",
    color="ai_mode",
    title="Wie viel könnte KI übernehmen?",
)
figure.update_traces(texttemplate="%{text} %", textposition="outside")
figure.update_layout(showlegend=False)
figure.show()

In [ ]:
# Bucket x ai_mode: zeigt, dass die beiden Achsen wirklich Unterschiedliches messen.
cross = pd.crosstab(labelled["resolution_bucket"], labelled["ai_mode"])
display(cross)
if not cross.empty:
    figure = px.imshow(
        cross,
        text_auto=True,
        aspect="auto",
        color_continuous_scale="Blues",
        labels={"x": "ai_mode", "y": "Bucket", "color": "Tickets"},
        title="Auflösungs-Bucket × KI-Eignung",
    )
    figure.show()

### 6.1 · Kategorien — das eigentliche Ergebnis

Welche Hauptkategorien bestehen überwiegend aus beantwortbaren Fragen und Routine-Aufgaben,
und welche aus echten Eskalationen? Das ist die Antwort auf „wo setzen wir KI zuerst an“.

In [ ]:
TOP_N_CATEGORIES = 20

top_categories = labelled["Hauptkategorie"].value_counts().head(TOP_N_CATEGORIES).index
subset = labelled[labelled["Hauptkategorie"].isin(top_categories)]

share = (
    pd.crosstab(
        subset["Hauptkategorie"], subset["resolution_bucket"], normalize="index"
    )
    * 100
).round(1)
share = share.reindex(index=top_categories, columns=BUCKET_ORDER, fill_value=0)

figure = px.imshow(
    share,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Bucket", "y": "Hauptkategorie", "color": "% der Kategorie"},
    title=f"Auflösungs-Bucket je Hauptkategorie (Top {TOP_N_CATEGORIES}, zeilenweise %)",
)
figure.update_layout(height=620)
figure.show()

In [ ]:
category_summary = (
    labelled.groupby("Hauptkategorie", observed=True)
    .agg(
        Tickets=("key", "size"),
        frage_antwort=("resolution_bucket", lambda s: (s == "question_answer").mean()),
        support_todo=("resolution_bucket", lambda s: (s == "support_todo").mean()),
        team_handoff=("resolution_bucket", lambda s: (s == "team_handoff").mean()),
        voll_automatisierbar=("ai_mode", lambda s: (s == "fully_automatable").mean()),
        ki_entwurf=("ai_mode", lambda s: (s == "ai_drafts_human_sends").mean()),
    )
    .reset_index()
)
category_summary["KI_adressierbar"] = (
    category_summary["voll_automatisierbar"] + category_summary["ki_entwurf"]
)
category_summary["KI_adressierbare_Tickets"] = (
    category_summary["KI_adressierbar"] * category_summary["Tickets"]
).round(0)

for column in [
    "frage_antwort",
    "support_todo",
    "team_handoff",
    "voll_automatisierbar",
    "ki_entwurf",
    "KI_adressierbar",
]:
    category_summary[column] = (category_summary[column] * 100).round(1)

category_summary = category_summary.sort_values(
    "KI_adressierbare_Tickets", ascending=False
)
display(category_summary.head(25))

### 6.2 · Kleinere Buckets — was sich aus den Intents ergibt

`intent_label` war im Prompt frei. Was hier oben steht, sind die real wiederkehrenden
Anfragetypen — die Kandidaten für einzelne KI-Automatisierungen, jeweils mit dem Anteil,
den das Modell für voll automatisierbar hält.

In [ ]:
MIN_INTENT_COUNT = 5

intents = (
    labelled.dropna(subset=["intent_label"])
    .groupby("intent_label", observed=True)
    .agg(
        Tickets=("key", "size"),
        Bucket=(
            "resolution_bucket",
            lambda s: s.mode().iloc[0] if len(s.mode()) else None,
        ),
        voll_automatisierbar=("ai_mode", lambda s: (s == "fully_automatable").mean()),
        Top_Kategorie=(
            "Hauptkategorie",
            lambda s: s.mode().iloc[0] if len(s.mode()) else None,
        ),
    )
    .reset_index()
)
intents["voll_automatisierbar"] = (intents["voll_automatisierbar"] * 100).round(1)
intents = intents[intents["Tickets"] >= MIN_INTENT_COUNT].sort_values(
    "Tickets", ascending=False
)

print(
    f"{len(intents)} wiederkehrende Intents mit >= {MIN_INTENT_COUNT} Tickets, "
    f"zusammen {intents['Tickets'].sum():,} Tickets "
    f"({intents['Tickets'].sum() / max(len(labelled), 1) * 100:.0f} % des Aufkommens)"
)
display(intents.head(40))

In [ ]:
top_intents = intents.head(25).sort_values("Tickets")
figure = px.bar(
    top_intents,
    x="Tickets",
    y="intent_label",
    color="Bucket",
    orientation="h",
    title="Top 25 wiederkehrende Anfragetypen",
)
figure.update_layout(height=700, yaxis_title=None)
figure.show()

### 6.3 · Aufwand statt Stückzahl

Tickets sind nicht gleich teuer. Gewichtet man das Volumen mit der tatsächlichen
Bearbeitungsdauer (Business-Stunden), verschiebt sich das Bild — ein seltener, aber teurer
Anfragetyp kann wichtiger sein als ein häufiger Einzeiler.

In [ ]:
labelled["biz_hours"] = pd.to_numeric(
    labelled["time_to_resolution_biz_hours"], errors="coerce"
)

effort = (
    labelled.dropna(subset=["ai_mode"])
    .groupby("ai_mode", observed=True)
    .agg(
        Tickets=("key", "size"),
        Median_Stunden=("biz_hours", "median"),
        Summe_Stunden=("biz_hours", "sum"),
    )
    .reset_index()
)
effort["Anteil Tickets %"] = (effort["Tickets"] / effort["Tickets"].sum() * 100).round(
    1
)
effort["Anteil Aufwand %"] = (
    effort["Summe_Stunden"] / effort["Summe_Stunden"].sum() * 100
).round(1)
effort["Median_Stunden"] = effort["Median_Stunden"].round(2)
effort["Summe_Stunden"] = effort["Summe_Stunden"].round(0)
display(effort)

print(
    "\nHinweis: time_to_resolution_biz_hours misst die Zeit bis zum Statuswechsel, nicht die\n"
    "reine Bearbeitungszeit am Ticket. Als Groessenordnung brauchbar, nicht als Kostenrechnung."
)

## 7 · Management-Summary

In [ ]:
total = len(labelled)
labelled_ok = labelled.dropna(subset=["resolution_bucket"])


def share(mask):
    """Return percent of the selected cohort with an explicit empty denominator."""
    return mask.sum() / max(total, 1) * 100


lines = [
    f"**Datenbasis:** {total:,} Amparex-Tickets (SDAX), "
    f"{START_DATE} bis {END_DATE}; {int(success.sum())} mit aktuellem Ergebnis.",
    "",
    "**Auflösung**",
]
for bucket in BUCKET_ORDER:
    count = (labelled["resolution_bucket"] == bucket).sum()
    lines.append(
        f"- `{bucket}`: {count:,} Tickets ({count / max(total, 1) * 100:.1f} %)"
    )

lines += ["", "**KI-Eignung**"]
for mode in AI_MODE_ORDER:
    count = (labelled["ai_mode"] == mode).sum()
    lines.append(f"- `{mode}`: {count:,} Tickets ({count / max(total, 1) * 100:.1f} %)")

addressable = (
    labelled["ai_mode"].isin(["fully_automatable", "ai_drafts_human_sends"]).sum()
)
lines += [
    "",
    f"**KI-adressierbar gesamt:** {addressable:,} Tickets "
    f"({addressable / max(total, 1) * 100:.1f} % des Aufkommens) — voll automatisierbar oder "
    "KI-Entwurf mit menschlicher Freigabe.",
    "",
    "**Wo zuerst ansetzen** (Hauptkategorien nach KI-adressierbarem Volumen):",
]
for _, row in category_summary.head(5).iterrows():
    lines.append(
        f"- **{row['Hauptkategorie']}** — {int(row['Tickets']):,} Tickets, "
        f"davon {row['KI_adressierbar']:.0f} % KI-adressierbar "
        f"({int(row['KI_adressierbare_Tickets']):,} Tickets)"
    )

lines += ["", "**Häufigste einzelne Anfragetypen:**"]
for _, row in intents.head(5).iterrows():
    lines.append(
        f'- „{row["intent_label"]}" — {int(row["Tickets"]):,} Tickets, '
        f"{row['voll_automatisierbar']:.0f} % voll automatisierbar"
    )

display(Markdown("\n".join(lines)))

In [ ]:
# Explicit aggregate export only; no files are written by default.
if OUTPUT_DIR is not None:
    output = Path(OUTPUT_DIR).resolve()
    output.mkdir(parents=True, exist_ok=True)
    (output / "analysis-manifest.json").write_text(
        json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
    )
    category_summary.to_csv(output / "by-category.csv", sep=";", index=False)
    intents.to_csv(output / "intents.csv", sep=";", index=False)
    print("Aggregierte Exporte und Manifest im gewaehlten OUTPUT_DIR gespeichert.")
else:
    print("OUTPUT_DIR=None: keine Exporte geschrieben.")